# SparkyLLM local_agent — Colab deploy

Runs `local_agent/app.py` (Streamlit + ReAct-lite agent) on a Colab GPU and exposes it via a Cloudflare Quick Tunnel.

**Why Cloudflare instead of ngrok?** No account, no auth token, no orphaned-endpoint problem on the free tier. Each run gets a fresh ephemeral `*.trycloudflare.com` URL.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**
2. Make sure Drive has these files at the canonical paths:
   - `MyDrive/sparkyllm/checkpoints/gpt_medium_dpo_best.pth`
   - `MyDrive/sparkyllm/datasets_pretrain/tokenizer_out/tokenizer.json`

Then just run all cells.

## 1. Install dependencies

In [ ]:
!pip install -q streamlit tokenizers
# torch is already installed in Colab

## 2. Clone the sparkyllm repo (or pull latest)

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ajencinas/sparkyllm.git'
REPO_DIR = '/content/sparkyllm'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo exists, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

%cd {REPO_DIR}
!ls

## 3. Mount Drive and copy weights

Copies the DPO checkpoint + tokenizer into `local_test/weights/` (which is what `local_agent/app.py` reads from).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

SPARKYLLM_DRIVE = '/content/drive/MyDrive/sparkyllm'
WEIGHTS_DIR = os.path.join(REPO_DIR, 'local_test', 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

files_to_copy = [
    ('checkpoints/gpt_medium_dpo_best.pth', 'gpt_medium_dpo_best.pth'),
    # tools checkpoint is optional — copy it if it exists, skip silently otherwise
    ('checkpoints/gpt_medium_tools_best.pth', 'gpt_medium_tools_best.pth'),
    ('datasets_pretrain/tokenizer_out/tokenizer.json', 'tokenizer.json'),
]

for src_rel, dst_name in files_to_copy:
    src = os.path.join(SPARKYLLM_DRIVE, src_rel)
    dst = os.path.join(WEIGHTS_DIR, dst_name)
    if not os.path.exists(src):
        print(f'  MISSING on Drive: {src}')
        continue
    if os.path.exists(dst):
        print(f'  already in place: {dst_name}')
        continue
    print(f'  copying {dst_name}...')
    shutil.copy2(src, dst)

print()
print('Weights folder contents:')
for f in sorted(os.listdir(WEIGHTS_DIR)):
    sz = os.path.getsize(os.path.join(WEIGHTS_DIR, f)) / 1024 / 1024
    print(f'  {f}  ({sz:.1f} MB)')

## 4. Install cloudflared

Downloads the Cloudflare tunnel binary (~30 MB). One-time per Colab session — re-running the cell is a no-op if it's already there.

In [ ]:
CLOUDFLARED_BIN = '/usr/local/bin/cloudflared'

if not os.path.exists(CLOUDFLARED_BIN):
    print('Downloading cloudflared...')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', CLOUDFLARED_BIN,
    ], check=True)
    subprocess.run(['chmod', '+x', CLOUDFLARED_BIN], check=True)
    print('cloudflared installed')
else:
    print('cloudflared already installed')

subprocess.run([CLOUDFLARED_BIN, '--version'])

## 5. Launch Streamlit + open tunnel

Starts Streamlit headless on port 8501, then exposes it via a Cloudflare Quick Tunnel. The public URL is printed at the bottom.

**Keep this notebook running while you use the app.** Closing the tab or letting Colab idle will tear down both processes.

In [ ]:
import time, re

APP_PATH = os.path.join(REPO_DIR, 'local_agent', 'app.py')
PORT = 8501

# 5a. Start Streamlit in the background
streamlit_proc = subprocess.Popen(
    [
        'streamlit', 'run', APP_PATH,
        '--server.port', str(PORT),
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--browser.gatherUsageStats', 'false',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print('Starting Streamlit...')
ready = False
for _ in range(60):
    line = streamlit_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    print('  ' + line.rstrip())
    if 'You can now view' in line or f'http://localhost:{PORT}' in line:
        ready = True
        break

if not ready:
    print('WARNING: Streamlit did not signal ready within 60s. Proceeding anyway.')

# 5b. Start cloudflared Quick Tunnel pointing at Streamlit
print()
print('Starting Cloudflare tunnel...')
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED_BIN, 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url_re = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
public_url = None
for _ in range(120):  # cloudflared usually prints the URL within a few seconds
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    if 'trycloudflare.com' in line or 'Your quick Tunnel' in line:
        print('  ' + line.rstrip())
    m = url_re.search(line)
    if m:
        public_url = m.group(0)
        break

print()
if public_url:
    print('=' * 60)
    print(f'  Public URL: {public_url}')
    print('=' * 60)
    print()
    print('Click the URL above to use the agent.')
    print('It may take 10-20 seconds before the URL is reachable on first visit.')
    print()
    print('To stop everything cleanly, run the next cell.')
else:
    print('WARNING: did not detect a tunnel URL — check the output above.')

## 6. Stop the server

Run this to cleanly tear down the tunnel and Streamlit. (Optional — closing the notebook does it too.)

In [ ]:
for name, proc in [('cloudflared', tunnel_proc), ('streamlit', streamlit_proc)]:
    try:
        proc.terminate()
        proc.wait(timeout=5)
        print(f'{name} stopped')
    except Exception as e:
        print(f'{name} cleanup: {e}')

## Troubleshooting

- **`ModuleNotFoundError` on first agent prompt**: cell 1 install probably hadn't finished. Re-run cell 1, then cell 5.
- **`Cannot reach API at ...`** or 502 from the tunnel: Streamlit didn't start. Look at cell 5 output for the actual Streamlit error.
- **Tunnel URL printed but the page won't load**: give it 10-20 seconds. Cloudflare's edge needs a moment to register the new tunnel before traffic flows.
- **Out of memory**: pick a Colab runtime with more RAM, or restart the runtime to free leaked allocations.
- **Tunnel disconnects mid-session**: Quick Tunnels die with the cloudflared process. Re-run cell 5 to relaunch (Streamlit can stay running, cloudflared will reconnect to it).
- **Want a stable URL**: Quick Tunnels are ephemeral by design. For a persistent URL you'd need a Cloudflare account + a named Tunnel — out of scope for this notebook.
- **Model is slow even on GPU**: confirm Runtime → Change runtime type is set to T4 GPU. The agent's first request is slowest because of model load + Python warmup.